# Analisis de Resultados ETL - Vehiculos Electricos

Este notebook realiza:
1. Lectura de tablas Silver y Golden desde Unity Catalog.
2. Validaciones de conteo y cobertura del join.
3. Analisis de agregaciones de negocio.
4. Visualizaciones para insights finales.

In [ ]:
dbutils.widgets.removeAll()

dbutils.widgets.text("catalog_name", "catalog_smartdata_final")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("golden_schema", "golden")

import matplotlib.pyplot as plt
from pyspark.sql import functions as F

NameError: name 'dbutils' is not defined

In [ ]:
catalog_name = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")
golden_schema = dbutils.widgets.get("golden_schema")

golden_db = f"{catalog_name}.{golden_schema}"
silver_db = f"{catalog_name}.{silver_schema}"

tbl_popularity = f"{golden_db}.golden_ev_model_popularity"
tbl_price_range = f"{golden_db}.golden_make_price_range"
tbl_state = f"{golden_db}.golden_state_distribution"
tbl_range_regs = f"{golden_db}.golden_range_vs_registrations"
tbl_ratio = f"{golden_db}.golden_make_range_price_ratio"
tbl_enriched = f"{silver_db}.ev_vehicle_enriched"

print(f"Golden DB: {golden_db}")
print(f"Silver DB: {silver_db}")

In [ ]:
print("Tablas disponibles en GOLDEN:")
display(spark.sql(f"SHOW TABLES IN {golden_db}"))

print("Tablas disponibles en SILVER:")
display(spark.sql(f"SHOW TABLES IN {silver_db}"))

In [ ]:
df_popularity = spark.table(tbl_popularity)
df_price_range = spark.table(tbl_price_range)
df_state = spark.table(tbl_state)
df_range_regs = spark.table(tbl_range_regs)
df_ratio = spark.table(tbl_ratio)

display(df_popularity.orderBy(F.desc("vehicles_registered")).limit(20))
display(df_price_range.orderBy(F.desc("avg_range")).limit(20))

In [ ]:
# Cobertura del join en Silver
df_enriched = spark.table(tbl_enriched)

join_metrics = (
    df_enriched.agg(
        F.count(F.lit(1)).alias("total_registros"),
        F.sum(F.when(F.col("has_specs_match"), 1).otherwise(0)).alias("registros_con_match")
    )
    .withColumn("porcentaje_match", F.round(F.col("registros_con_match") / F.col("total_registros") * 100, 2))
)

display(join_metrics)

In [ ]:
top_models = df_popularity.orderBy(F.desc("vehicles_registered")).limit(15).toPandas()
top_states = df_state.orderBy(F.desc("total_ev")).limit(15).toPandas()
make_price_range = df_price_range.orderBy(F.desc("total_vehicles")).limit(20).toPandas()
range_regs = df_range_regs.filter(F.col("specs_range_km").isNotNull()).toPandas()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

axes[0, 0].barh(top_models["make"] + " - " + top_models["model"], top_models["vehicles_registered"])
axes[0, 0].invert_yaxis()
axes[0, 0].set_title("Top 15 modelos por registros")
axes[0, 0].set_xlabel("Vehiculos registrados")

axes[0, 1].barh(top_states["state"], top_states["total_ev"], color="tab:green")
axes[0, 1].invert_yaxis()
axes[0, 1].set_title("Distribucion geografica (Top 15 estados)")
axes[0, 1].set_xlabel("Total EV")

axes[1, 0].scatter(make_price_range["avg_price"], make_price_range["avg_range"], alpha=0.8, color="tab:orange")
for _, row in make_price_range.head(10).iterrows():
    axes[1, 0].annotate(row["make"], (row["avg_price"], row["avg_range"]))
axes[1, 0].set_title("Precio promedio vs autonomia promedio")
axes[1, 0].set_xlabel("Precio promedio")
axes[1, 0].set_ylabel("Autonomia promedio")

axes[1, 1].scatter(range_regs["specs_range_km"], range_regs["vehicles_registered"], alpha=0.5, color="tab:red")
axes[1, 1].set_title("Autonomia del modelo vs registros")
axes[1, 1].set_xlabel("Autonomia especificada (km)")
axes[1, 1].set_ylabel("Vehiculos registrados")

plt.tight_layout()
plt.show()

In [ ]:
top_model = df_popularity.orderBy(F.desc("vehicles_registered")).first()
top_state = df_state.orderBy(F.desc("total_ev")).first()
best_ratio = df_ratio.orderBy(F.desc("range_per_price_ratio")).first()

print("INSIGHTS CLAVE")
print(f"1) Modelo lider en registros: {top_model['make']} {top_model['model']} ({top_model['vehicles_registered']})")
print(f"2) Estado con mayor adopcion: {top_state['state']} ({top_state['total_ev']})")
print(f"3) Mejor relacion rango/precio (promedio marca): {best_ratio['make']} ({round(best_ratio['range_per_price_ratio'], 6)})")
print("4) La tabla golden_range_vs_registrations permite validar si mayor autonomia implica mas registros por modelo.")